# Unit 2, Lecture 4: Workflow chaining and event-driven systems

One tool call was the atom. Now we build the molecule. Two shapes:

- a **pipeline**: named steps in order, each reading what the last one produced
- an **event bus**: publish an event, and whoever subscribed reacts

The example stays support-ticket triage from Lecture 1, grown from one routing
decision into a real intake pipeline: validate, classify, enrich, assign.

Everything here is pure Python, no model, so it runs anywhere.

## Setup

In [ ]:
from cse476.pipeline import (
    build_intake_pipeline, Pipeline, StepResult,
    validate_ticket, classify_ticket, enrich_ticket, assign_ticket,
    EventBus, Event,
)

pipeline = build_intake_pipeline()
print("steps:", [name for name, _ in pipeline._steps])

## 1. A full run: state flows down the chain

Watch a ticket move through all four steps. Each step reads what earlier steps
left in the state and adds its own contribution.

In [ ]:
state, log = pipeline.run({"text": "I was charged twice and want a refund."})
print(log)
print()
print("final state:")
for k, v in state.items():
    print(f"  {k}: {v}")

`enrich` could only attach the SLA because `classify` had already put `queue`
into the state. That dependency is the whole point of a chain: order matters
because each step builds on the last.

## 2. The rule that makes pipelines safe

When a step fails, the pipeline **stops** and never runs the next step on a
half-finished state. Feed an empty ticket and watch it stop at step one, without
inventing a queue or assigning a nonexistent team.

In [ ]:
state, log = pipeline.run({"text": ""})
print(log)
print()
print("queue invented?", "queue" in state)
print("team assigned?  ", "assigned_to" in state)

Nothing downstream ran. No wrong answer was produced. **A clean stop with a
clear reason beats a full run with a confident, wrong result.**

## 3. Stopping keeps the work that already succeeded

Make a middle step fail on purpose. The steps before it keep their results; the
steps after it never run.

In [ ]:
def failing_enrich(state):
    return StepResult(ok=False, note="pretend the SLA service is down")

partial = (
    Pipeline("partial")
    .step("validate", validate_ticket)
    .step("classify", classify_ticket)
    .step("enrich", failing_enrich)
    .step("assign", assign_ticket)
)

state, log = partial.run({"text": "my login is broken"})
print(log)
print()
print("classify's work survived:", state.get("queue"))
print("assign never ran:       ", "assigned_to" not in state)

The `log` is a first taste of **observability**, Unit 5's big theme. A system
that can say exactly what it did and where it stopped is a system you can
actually debug. The habit starts here: every run explains itself.

## 4. The event-driven shape

A pipeline is a line you walk down. An event bus is a room you shout into: you
announce that something happened, and whoever was listening reacts. The emitter
does not know who listens, so you can add reactions later without touching the
trigger.

In [ ]:
bus = EventBus()

bus.on("ticket.created", lambda e: print(f"  logger:   recorded ticket {e.payload['id']}"))
bus.on("ticket.created", lambda e: print(f"  notifier: emailed the team about {e.payload['id']}"))
bus.on("ticket.created", lambda e: print(f"  metrics:  incremented the counter"))

reacted = bus.emit(Event("ticket.created", {"id": 7}))
print(f"\n{reacted} handlers reacted, and none of them knows about the others.")

## 5. One broken listener does not silence the rest

In an event system, reactions are independent. If one handler throws, the others
must still run.

In [ ]:
bus2 = EventBus()

def broken(e):
    raise RuntimeError("email server down")

survived = []
bus2.on("x", broken)
bus2.on("x", lambda e: survived.append("metrics still ran"))

reacted = bus2.emit(Event("x"))
print("survivors:", survived)
print("handlers that completed:", reacted, "of 2")

## 6. They compose

The two shapes are not rivals. A common pattern: a pipeline finishes its ordered
work, then **emits an event** so independent handlers can react to the result.

In [ ]:
bus = EventBus()
bus.on("ticket.assigned", lambda e: print(f"  -> notifying {e.payload['assigned_to']}"))
bus.on("ticket.assigned", lambda e: print(f"  -> logging priority: {e.payload['priority']}"))

state, log = build_intake_pipeline().run({"text": "someone hacked my account"})
if log.completed:
    print("pipeline done, now the reactions:")
    bus.emit(Event("ticket.assigned", {
        "assigned_to": state["assigned_to"],
        "priority": state["priority"],
    }))

## Your turn

**1. Add a fifth step.** Insert a `deduplicate` step that stops the run if the
same ticket text was already seen. Notice you only touch one line, `.step(...)`,
and nothing else in the pipeline changes. That ease of change is the whole payoff.

**2. Break a middle step.** Make `classify` fail and confirm the pipeline stops
there, keeps the `validate` result, and never assigns a ticket to a nonexistent
queue. Read the log.

**3. Pipeline or events?** For your capstone, name one part that is genuinely
ordered steps, and one part that is really independent reactions to an event. Do
not force everything into one shape.

In [ ]:
# your work here
